In [ ]:
import os
import glob
import pandas as pd
import re

def parse_logs_to_csv(output_filename="mental_qa_results.csv"):
    # 현재 디렉토리의 모든 txt 파일 찾기
    log_files = glob.glob("./logs/*.txt")
    
    all_data = []

    print(f"Found {len(log_files)} log files. Processing...")

    for filepath in log_files:
        filename = os.path.basename(filepath)
        
        # 1. 파일명 파싱 (예: Qwen_Qwen2.5-14B-Instruct_low.txt)
        # 마지막 '_'를 기준으로 모델명과 조건(low/medium)을 분리합니다.
        try:
            name_part, extension = os.path.splitext(filename)
            if '_' in name_part:
                model_name, condition = name_part.rsplit('_', 1)
            else:
                model_name = name_part
                condition = "unknown"
        except Exception as e:
            print(f"Skipping filename parsing for {filename}: {e}")
            continue

        # 2. 파일 내용 읽기
        with open(filepath, 'r', encoding='utf-8') as f:
            lines = f.readlines()

        # 3. 테이블 데이터 파싱
        # 파이프(|)로 구분된 테이블 행을 찾습니다.
        for line in lines:
            line = line.strip()
            
            # 데이터 행인지 확인 (D로 시작하는 질병 코드가 있는지, 파이프로 시작하는지)
            # 예: | D001       |       60 |       15 |       75 |     80.00% |
            if line.startswith('|') and 'D' in line:
                parts = [p.strip() for p in line.split('|')]
                
                # parts[0]은 빈 문자열(시작 파이프 앞), parts[1]이 Disease, ...
                # 유효한 데이터 행의 길이는 보통 7개입니다 ('', Disease, Correct, Wrong, Total, Acc, '')
                if len(parts) >= 6:
                    disease_code = parts[1]
                    
                    # 헤더나 구분선, TOTAL 행 제외
                    if not disease_code.startswith('D'):
                        continue
                        
                    try:
                        correct = int(parts[2])
                        wrong = int(parts[3])
                        total = int(parts[4])
                        accuracy_str = parts[5].replace('%', '')
                        accuracy = float(accuracy_str)

                        all_data.append({
                            "Model": model_name,
                            "Condition": condition, # low / medium
                            "Disease": disease_code,
                            "Correct": correct,
                            "Wrong": wrong,
                            "Total": total,
                            "Accuracy": accuracy,
                            "Filename": filename
                        })
                    except ValueError:
                        continue

    # 데이터프레임 생성
    df = pd.DataFrame(all_data)
    
    # 보기 좋게 정렬 (모델명 -> 조건 -> 질병코드 순)
    if not df.empty:
        df = df.sort_values(by=["Model", "Condition", "Disease"])
        
        # CSV 저장
        df.to_csv(output_filename, index=False, encoding='utf-8-sig')
        print(f"\nSuccess! Saved to {output_filename}")
        print(f"Total rows extracted: {len(df)}")
        print(df.head()) # 미리보기 출력
    else:
        print("No data extracted. Please check the log file format.")

if __name__ == "__main__":
    parse_logs_to_csv()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

csv_file="mental_qa_results.csv"
try:
    df = pd.read_csv(csv_file)
except FileNotFoundError:
    print(f"Error: {csv_file} not found.")

# 2. Pivot
df_low = df[df['Condition'] == 'low'].pivot(index='Model', columns='Disease', values='Accuracy')
df_medium = df[df['Condition'] == 'medium'].pivot(index='Model', columns='Disease', values='Accuracy')

# if not df_low.empty:
#     df_low['Average'] = df_low.mean(axis=1)

# if not df_medium.empty:
#     df_medium['Average'] = df_medium.mean(axis=1)

# 3. 모델 필터링 리스트
desired_order = [
    # Qwen 2.5 Series
    # "Qwen2.5-3B-Instruct",
    # "Qwen2.5-7B-Instruct",
    "Qwen_Qwen2.5-14B-Instruct",
    # "Qwen2.5-32B-Instruct",
    "Qwen_Qwen2.5-72B-Instruct",
    
    # Qwen 3 Series
    # "Qwen3-4B", 
    "Qwen_Qwen3-8B",
    # "Qwen3-14B",
    "Qwen_Qwen3-32B",

    # Gemma Series
    # "gemma-3-4b-it",
    "google_gemma-3-12b-it",
    "google_gemma-3-27b-it",

    # Llama Series
    "meta-llama_Llama-3.1-8B-Instruct",
    "meta-llama_Llama-3.1-70B-Instruct",

    # MentaLLaMA Series
    "klyang_MentaLLaMA-chat-7B",
    "klyang_MentaLLaMA-chat-13B",

    # GPT
    # "gpt-4o",
    # "gpt-5-mini",
    "gpt-5.1",

    # "gemini-2.5-flash",
    "google_gemini-2.5-pro",

    #
    # "claude-haiku-4.5",
    "anthropic_claude-sonnet-4.5",


]

# 3. 필터링 및 정렬 적용 함수
def filter_and_sort_dataframe(data_df, order_list):
    valid_order = [m for m in order_list if m in data_df.index]
    return data_df.reindex(valid_order)

df_low_sorted = filter_and_sort_dataframe(df_low, desired_order)
df_medium_sorted = filter_and_sort_dataframe(df_medium, desired_order)

# === [HELPER] 모델의 시리즈(계열)을 판별하는 함수 ===
def get_series_key(model_name):
    name = model_name.lower()
    if 'qwen2.5' in name: return 'qwen2.5'
    if 'qwen3' in name: return 'qwen3'
    if 'gemma' in name: return 'gemma'
    if 'mentallama' in name: return 'mentallama' # llama보다 먼저 체크해야 함 (이름 겹침 방지)
    if 'llama' in name: return 'llama'
    if 'gpt' in name: return 'gpt'
    if 'gemini' in name: return 'gemini'
    if 'claude' in name: return 'claude'
    return 'other'

# 4. 시각화 함수 (수정됨: 굵은 선 추가 기능 포함)
def plot_heatmap(data, title):
    if data.empty:
        print(f"No data available for {title}.")
        return

    plt.figure(figsize=(12, 5.5))
    
    # 히트맵 그리기
    ax = sns.heatmap(data, annot=True, fmt=".1f", cmap="RdYlGn", linewidths=.5,annot_kws={"size": 6.5} )
    
    plt.title(title, fontsize=18)
    plt.xlabel("Disease Code", fontsize=12)
    plt.ylabel(" ", fontsize=12)
    plt.xticks(rotation=45)

    # === [ADDITION] 시리즈 사이에 굵은 선 긋기 ===
    # 데이터프레임의 인덱스를 순회하며 다음 모델과 시리즈가 다른지 확인
    models = data.index.tolist()
    for i in range(len(models) - 1):
        current_series = get_series_key(models[i])
        next_series = get_series_key(models[i+1])
        
        # 시리즈가 달라지는 지점에 선 긋기
        if current_series != next_series:
            # y좌표는 위에서부터 0, 1, 2... 이므로 i+1 위치가 경계선
            ax.axhline(i + 1, color='black', linewidth=1.5) # linewidth로 굵기 조절
    
    plt.tight_layout()
    plt.savefig(f"{title.replace(' ', '_')}.png")
    plt.show()

# 5. 출력
print("=== Processing Low Condition (Filtered) ===")
if not df_low_sorted.empty:
    plot_heatmap(df_low_sorted, "Accuracy Heatmap - Type I")

print("\n=== Processing Medium Condition (Filtered) ===")
if not df_medium_sorted.empty:
    plot_heatmap(df_medium_sorted, "Accuracy Heatmap - Type II")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. 시각화용 데이터 준비 (D001~D023 컬럼만 선택)
disease_cols = [col for col in df_low_sorted.columns if col.startswith('D')]
df_trend = df_low_sorted[disease_cols].transpose()

# 2. 그래프 설정
plt.figure(figsize=(20, 8))

# 3. 모델별 선 그래프 그리기
# 투명도(alpha)를 조절하여 여러 선이 겹쳐도 흐름을 볼 수 있게 함
for model in df_trend.columns:
    plt.plot(df_trend.index, df_trend[model], 
             marker='o',          # 데이터 포인트 표시
             markersize=5, 
             linewidth=2, 
             alpha=0.5,           # 요청하신 0.3~0.5 수준의 투명도
             label=model)

# 4. 차트 상세 스타일링
plt.title('Accuracy Line Plot - Type I', fontsize=18, fontweight='bold', pad=20)
plt.xlabel('Disease ID (Progression)', fontsize=14)
plt.ylabel('Score (0-100)', fontsize=14)

# X축 라벨 가독성 높이기
plt.xticks(rotation=0)
plt.grid(True, axis='both', linestyle='--', alpha=0.3)

# 범례 설정 (차트 바깥쪽 우측에 배치)
plt.legend(title='Models', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=10)

# Y축 범위 고정
plt.ylim(-5, 105)

# 5. 레이아웃 조정 및 출력
plt.tight_layout()
plt.show()

In [ ]:
import os
import glob
import pandas as pd
import re

def parse_new_logs_to_csv(log_dir, output_filename="mental_qa_type_results.csv"):
    # 로그 파일 경로 설정
    log_files = glob.glob(os.path.join(log_dir, "*.txt"))
    all_data = []

    print(f"Found {len(log_files)} log files. Processing...")

    for filepath in log_files:
        filename = os.path.basename(filepath)
        
        with open(filepath, 'r', encoding='utf-8') as f:
            lines = f.readlines()

        model_name = "unknown"
        difficulty = "unknown"
        table_count = 0  # 첫 번째 테이블은 Type 3, 두 번째는 Type 4로 구분하기 위함
        
        i = 0
        while i < len(lines):
            line = lines[i].strip()

            # 1. 모델명 및 난이도 추출
            if "Tested on:" in line:
                model_name = line.split("Tested on:")[1].strip()
            elif "Difficulty:" in line:
                difficulty = line.split("Difficulty:")[1].strip()

            # 2. 테이블 시작 감지 (| Disease | Correct | ...)
            if line.startswith('|') and 'Disease' in line:
                table_count += 1
                # 현재 테이블의 타입 결정 (첫 번째면 Type 3, 두 번째면 Type 4)
                current_type = f"Type{2 + table_count}" # 1번째면 Type3, 2번째면 Type4
                
                # 헤더와 구분선(---) 건너뛰기
                i += 2 
                
                # 데이터 행 읽기
                while i < len(lines):
                    data_line = lines[i].strip()
                    
                    # 테이블 끝 감지 (구분선이나 빈 줄)
                    if not data_line.startswith('|') or "TOTAL" in data_line:
                        break
                    
                    parts = [p.strip() for p in data_line.split('|')]
                    
                    # 유효한 데이터 행 (| D001-D005 | 40 | 35 | 75 | 53.33% |)
                    if len(parts) >= 6 and 'D' in parts[1]:
                        try:
                            disease_code = parts[1]
                            correct = int(parts[2])
                            wrong = int(parts[3])
                            total = int(parts[4])
                            accuracy = float(parts[5].replace('%', ''))

                            all_data.append({
                                "Model": model_name,
                                "Difficulty": difficulty,
                                "Type": current_type,
                                "Disease": disease_code,
                                "Correct": correct,
                                "Wrong": wrong,
                                "Total": total,
                                "Accuracy": accuracy,
                                "Filename": filename
                            })
                        except (ValueError, IndexError):
                            pass
                    i += 1
            i += 1

    # 데이터프레임 생성 및 저장
    df = pd.DataFrame(all_data)
    
    if not df.empty:
        # 정렬: 모델 -> 난이도 -> 타입 -> 질병코드
        df = df.sort_values(by=["Model", "Difficulty", "Type", "Disease"])
        df.to_csv(output_filename, index=False, encoding='utf-8-sig')
        print(f"\nSuccess! Saved to {output_filename}")
        print(f"Total rows extracted: {len(df)}")
        print(df.head())
    else:
        print("No data extracted. Please check the log file format.")

if __name__ == "__main__":
    # 로그 파일이 있는 폴더 경로를 입력하세요
    LOG_DIR_PATH = "./logs_normal" 
    parse_new_logs_to_csv(LOG_DIR_PATH, output_filename="mental_qa_type34_normal_results.csv")
    LOG_DIR_PATH = "./logs_single" 
    parse_new_logs_to_csv(LOG_DIR_PATH, output_filename="mental_qa_type34_single_results.csv")
    LOG_DIR_PATH = "./logs_clear" 
    parse_new_logs_to_csv(LOG_DIR_PATH, output_filename="mental_qa_type34_clear_results.csv")

Found 22 log files. Processing...

Success! Saved to mental_qa_type34_normal_results.csv
Total rows extracted: 3480
                          Model Difficulty   Type    Disease  Correct  Wrong  \
3045  Qwen/Qwen2.5-14B-Instruct       high  Type3  D001-D005       47     28   
3046  Qwen/Qwen2.5-14B-Instruct       high  Type3  D001-D008       33     42   
3047  Qwen/Qwen2.5-14B-Instruct       high  Type3  D001-D011       57     18   
3048  Qwen/Qwen2.5-14B-Instruct       high  Type3  D001-D013       48     27   
3049  Qwen/Qwen2.5-14B-Instruct       high  Type3  D002-D005       34     41   

      Total  Accuracy                            Filename  
3045     75     62.67  Qwen_Qwen2.5-14B-Instruct_high.txt  
3046     75     44.00  Qwen_Qwen2.5-14B-Instruct_high.txt  
3047     75     76.00  Qwen_Qwen2.5-14B-Instruct_high.txt  
3048     75     64.00  Qwen_Qwen2.5-14B-Instruct_high.txt  
3049     75     45.33  Qwen_Qwen2.5-14B-Instruct_high.txt  
Found 22 log files. Processing...

Success!

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re

# 1. 분석할 파일 목록 정의 (ls 출력 결과 반영)
# 파일이 있는 경로를 정확히 지정해주세요. (예: 현재 경로라면 './')
base_path = './logs_clear' 

files = {
    # Qwen 2.5 Series
    'Qwen2.5-3B': 'Qwen_Qwen2.5-3B-Instruct_high.txt',
    'Qwen2.5-7B': 'Qwen_Qwen2.5-7B-Instruct_high.txt',
    'Qwen2.5-14B': 'Qwen_Qwen2.5-14B-Instruct_high.txt',
    'Qwen2.5-32B': 'Qwen_Qwen2.5-32B-Instruct_high.txt',
    'Qwen2.5-72B': 'Qwen_Qwen2.5-72B-Instruct_high.txt',
    
    # Qwen 3 Series
    'Qwen3-4B': 'Qwen_Qwen3-4B_high.txt',
    'Qwen3-8B': 'Qwen_Qwen3-8B_high.txt',
    'Qwen3-14B': 'Qwen_Qwen3-14B_high.txt',
    'Qwen3-32B': 'Qwen_Qwen3-32B_high.txt',
    
    # Google Gemini / Gemma
    'Gemma-3-4B': 'google_gemma-3-4b-it_high.txt',
    'Gemma-3-12B': 'google_gemma-3-12b-it_high.txt',
    'Gemma-3-27B': 'google_gemma-3-27b-it_high.txt',
    'Gemini-2.5-Flash': 'google_gemini-2.5-flash_high.txt',
    'Gemini-2.5-Pro': 'google_gemini-2.5-pro_high.txt',

    # Anthropic Claude
    'Claude-Haiku-4.5': 'anthropic_claude-haiku-4.5_high.txt',
    'Claude-Sonnet-4.5': 'anthropic_claude-sonnet-4.5_high.txt',

    # Meta Llama
    'Llama-3.1-8B': 'meta-llama_Llama-3.1-8B-Instruct_high.txt',
    'Llama-3.1-70B': 'meta-llama_Llama-3.1-70B-Instruct_high.txt',
    'Llama-3.3-70B': 'meta-llama_Llama-3.3-70B-Instruct_high.txt',

    # MentaLLaMA
    'MentaLLaMA-7B': 'klyang_MentaLLaMA-chat-7B_high.txt',
    'MentaLLaMA-13B': 'klyang_MentaLLaMA-chat-13B_high.txt',
}

detailed_data = []
summary_data = []

def parse_file(filename, model_name):
    filepath = os.path.join(base_path, filename)
    if not os.path.exists(filepath):
        print(f"Warning: File not found - {filename}")
        return

    with open(filepath, 'r', encoding='utf-8') as f:
        content = f.read()

    # 테이블 헤더를 기준으로 구역 나누기
    # 첫번째 분할은 헤더 앞부분(로그), 두번째는 Type3 테이블, 세번째는 Type4 테이블일 것으로 추정
    parts = content.split('| Disease    |  Correct |    Wrong |    Total |   Accuracy |')
    
    # parts[0]은 로그이므로 제외, parts[1]은 Type3, parts[2]는 Type4
    if len(parts) < 3:
        print(f"Warning: Could not find two tables in {filename}")
        return

    # 순서대로 Type3, Type4 지정
    table_types = ['type3', 'type4']
    
    for i, part in enumerate(parts[1:]): # 0번 인덱스(로그) 건너뛰기
        if i >= len(table_types): break
        
        current_type = table_types[i]
        lines = part.strip().split('\n')
        
        for line in lines:
            line = line.strip()
            # 데이터 행 파싱 (파이프(|)와 퍼센트(%)가 모두 있는 줄만 유효 데이터로 간주)
            if '|' in line and '%' in line:
                # 파이프로 분리하고 공백 제거
                cols = [c.strip() for c in line.split('|') if c.strip()]
                
                # TOTAL 행 파싱
                if 'TOTAL' in cols[0]:
                    summary_data.append({
                        'Model': model_name,
                        'Type': current_type,
                        'Total_Correct': int(cols[1]),
                        'Total_Wrong': int(cols[2]),
                        'Total_Count': int(cols[3]),
                        'Accuracy': float(cols[4].strip('%'))
                    })
                # 개별 Disease 행 파싱 (컬럼이 5개인 경우)
                elif len(cols) == 5:
                    detailed_data.append({
                        'Model': model_name,
                        'Type': current_type,
                        'Disease': cols[0],
                        'Correct': int(cols[1]),
                        'Wrong': int(cols[2]),
                        'Total': int(cols[3]),
                        'Accuracy': float(cols[4].strip('%'))
                    })

# 2. 파일 파싱 실행
for model, filename in files.items():
    parse_file(filename, model)

# DataFrame 생성
df_detailed = pd.DataFrame(detailed_data)
df_summary = pd.DataFrame(summary_data)

if df_summary.empty:
    print("No data parsed. Please check file paths.")
    exit()

# 3. 모델 정렬 순서 정의 (가독성을 위해 시리즈별/크기별 정렬)
model_order = [
    'Qwen2.5-3B', 'Qwen2.5-7B', 'Qwen2.5-14B', 'Qwen2.5-32B', 'Qwen2.5-72B',
    'Qwen3-4B', 'Qwen3-8B', 'Qwen3-14B', 'Qwen3-32B',
    'Gemma-3-4B', 'Gemma-3-12B', 'Gemma-3-27B',
    'Gemini-2.5-Flash', 'Gemini-2.5-Pro',
    'Claude-Haiku-4.5', 'Claude-Sonnet-4.5',
    'Llama-3.1-8B', 'Llama-3.1-70B', 'Llama-3.3-70B',
    'MentaLLaMA-7B', 'MentaLLaMA-13B'
]

# 데이터프레임의 Model 컬럼을 Categorical 타입으로 변환하여 정렬 순서 적용
existing_models = [m for m in model_order if m in df_summary['Model'].unique()]
df_summary['Model'] = pd.Categorical(df_summary['Model'], categories=existing_models, ordered=True)
df_detailed['Model'] = pd.Categorical(df_detailed['Model'], categories=existing_models, ordered=True)

# 4. 시각화 생성

sns.set_theme(style="whitegrid")
plt.rcParams['font.family'] = 'sans-serif'

# 4-1. 전체 성능 비교 (Bar Chart)
plt.figure(figsize=(18, 8))
ax = sns.barplot(x='Model', y='Accuracy', hue='Type', data=df_summary, palette='viridis')
for i in ax.containers:
    ax.bar_label(i, fmt='%.1f', padding=3, fontsize=9)
plt.title('Overall Accuracy Comparison by Model', fontsize=16, fontweight='bold')
plt.ylabel('Accuracy (%)')
plt.xlabel('')
plt.xticks(rotation=45, ha='right')
plt.ylim(0, 100)
plt.legend(title='Evaluation Type', loc='upper left', bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.savefig('overall_accuracy_comparison.png', dpi=300)
plt.close()

print("Saved: overall_accuracy_comparison.png")

# 4-2. Type 3 히트맵 (Disease Pair vs Model)
df_type3 = df_detailed[df_detailed['Type'] == 'type3']
if not df_type3.empty:
    pivot_type3 = df_type3.pivot(index='Disease', columns='Model', values='Accuracy')
    
    plt.figure(figsize=(16, 20))
    sns.heatmap(pivot_type3, annot=True, fmt='.0f', cmap='RdYlGn', 
                cbar_kws={'label': 'Accuracy (%)'}, linewidths=.5)
    plt.title('Type 3: Accuracy Heatmap (Multiple Choice)', fontsize=16, fontweight='bold')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig('clear_heatmap_type3.png', dpi=300)
    plt.close()
    print("Saved: clear_heatmap_type3.png")

# 4-3. Type 4 히트맵 (Disease Pair vs Model)
df_type4 = df_detailed[df_detailed['Type'] == 'type4']
if not df_type4.empty:
    pivot_type4 = df_type4.pivot(index='Disease', columns='Model', values='Accuracy')
    
    plt.figure(figsize=(16, 20))
    sns.heatmap(pivot_type4, annot=True, fmt='.0f', cmap='RdYlGn', 
                cbar_kws={'label': 'Accuracy (%)'}, linewidths=.5)
    plt.title('Type 4: Accuracy Heatmap (Generation)', fontsize=16, fontweight='bold')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig('clear_heatmap_type4.png', dpi=300)
    plt.close()
    print("Saved: clear_heatmap_type4.png")

print("All visualizations generated successfully.")

Saved: overall_accuracy_comparison.png
Saved: clear_heatmap_type3.png
Saved: clear_heatmap_type4.png
All visualizations generated successfully.


In [3]:
import os
import json
import re
import pandas as pd

# ----------------------------
# 1. 분석 로직 (기존과 동일)
# ----------------------------

def extract_option_char(text):
    if not isinstance(text, str):
        return ""
    match = re.search(r'^\s*([A-E])', text.strip(), re.IGNORECASE)
    if match:
        return match.group(1).upper()
    return ""

def check_response_contains_answer(response, answer_char):
    if not response or not answer_char:
        return False
    
    response = response.upper()
    target = answer_char.upper()
    
    patterns = [
        rf"(^|\s|\[|\(){target}(\)|\.|\]|,|:|$)",
        rf"OPTION\s+{target}"
    ]
    for pat in patterns:
        if re.search(pat, response):
            return True
            
    if "&" in response or "AND" in response:
        tokens = re.findall(r'[A-E]', response)
        if target in tokens:
            return True

    if len(response.strip()) == 1 and response.strip() == target:
        return True

    return False

def analyze_aggregated_performance(model_dir):
    """
    특정 모델 디렉토리 내부를 순회하며 Type 4 데이터를 분석하여 DataFrame 반환
    """
    results = []

    for root, dirs, files in os.walk(model_dir):
        path_parts = root.split(os.sep)
        
        if 'type4' not in path_parts:
            continue
            
        try:
            type_idx = path_parts.index('type4')
            lower_code = path_parts[type_idx - 1]
            upper_code = path_parts[type_idx - 2]
        except (IndexError, ValueError):
            continue

        for file in files:
            if not file.endswith('.json'):
                continue
            
            sub_type = ""
            if file.startswith('a_main'):
                sub_type = 'a_main (Upper->Lower)'
            elif file.startswith('b_main'):
                sub_type = 'b_main (Lower->Upper)'
            else:
                continue 
            
            file_path = os.path.join(root, file)
            
            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                
                total_q = 0
                correct_q = 0
                
                for key, item in data.items():
                    total_q += 1
                    answer_text = item.get('answer', '')
                    response_text = item.get('response', '')
                    answer_char = extract_option_char(answer_text)
                    
                    if check_response_contains_answer(response_text, answer_char):
                        correct_q += 1
                
                if total_q > 0:
                    results.append({
                        'Upper Code': upper_code,
                        'Lower Code': lower_code,
                        'Sub Type': sub_type,
                        'Total Questions': total_q,
                        'Correct Count': correct_q
                    })
                    
            except Exception as e:
                print(f"Error reading {file_path}: {e}")

    df = pd.DataFrame(results)

    if df.empty:
        return df

    # 합산 및 정확도 계산
    df_agg = df.groupby(['Upper Code', 'Lower Code', 'Sub Type']).sum().reset_index()
    df_agg['Accuracy'] = (df_agg['Correct Count'] / df_agg['Total Questions']) * 100
    df_agg = df_agg.sort_values(by='Upper Code', ascending=True)

    return df_agg

# ----------------------------
# 2. 모델별 개별 저장 로직 (수정됨)
# ----------------------------

def process_all_models_save_individually(base_output_dir):
    # base_output_dir에 있는 폴더 목록 (Brand)
    brands = [d for d in os.listdir(base_output_dir) if os.path.isdir(os.path.join(base_output_dir, d))]
    
    saved_files_count = 0

    for brand in brands:
        brand_path = os.path.join(base_output_dir, brand)
        
        # 브랜드 폴더 안의 모델 목록
        models = [m for m in os.listdir(brand_path) if os.path.isdir(os.path.join(brand_path, m))]
        
        for model_name in models:
            model_path = os.path.join(brand_path, model_name)
            print(f"Analyzing: {brand} / {model_name} ...")
            
            # 1. 해당 모델 분석
            df_model = analyze_aggregated_performance(model_path)
            
            if not df_model.empty:
                # 2. 메타 데이터 추가 (선택 사항이지만 파일 내부에 정보가 있는 것이 좋음)
                df_model['Brand'] = brand
                df_model['Model'] = model_name
                
                # 컬럼 순서 재배치 (Brand, Model을 맨 앞으로)
                cols = ['Brand', 'Model'] + [c for c in df_model.columns if c not in ['Brand', 'Model']]
                df_model = df_model[cols]
                
                # 3. 개별 CSV 파일로 저장
                # 저장 경로: root_path/모델이름.csv
                csv_filename = f"{model_name}.csv"
                save_path = os.path.join(base_output_dir, csv_filename)
                
                df_model.to_csv(save_path, index=False)
                print(f"  -> Saved: {save_path}")
                saved_files_count += 1
            else:
                print(f"  -> Skipped (No Data): {model_name}")

    print(f"\nProcessing Complete. Total {saved_files_count} CSV files created in {base_output_dir}")

# ----------------------------
# 3. 실행
# ----------------------------

root_path = "../output_clear"

process_all_models_save_individually(root_path)

Analyzing: meta-llama / Llama-3.1-8B-Instruct ...
  -> Saved: ../output_clear/Llama-3.1-8B-Instruct.csv
Analyzing: meta-llama / Llama-3.1-70B-Instruct ...
  -> Saved: ../output_clear/Llama-3.1-70B-Instruct.csv
Analyzing: anthropic / claude-haiku-4.5 ...
  -> Saved: ../output_clear/claude-haiku-4.5.csv
Analyzing: anthropic / claude-sonnet-4.5 ...
  -> Saved: ../output_clear/claude-sonnet-4.5.csv
Analyzing: Qwen / Qwen3-32B ...
  -> Saved: ../output_clear/Qwen3-32B.csv
Analyzing: Qwen / Qwen2.5-7B-Instruct ...
  -> Saved: ../output_clear/Qwen2.5-7B-Instruct.csv
Analyzing: Qwen / Qwen2.5-32B-Instruct ...
  -> Saved: ../output_clear/Qwen2.5-32B-Instruct.csv
Analyzing: Qwen / Qwen3-8B ...
  -> Saved: ../output_clear/Qwen3-8B.csv
Analyzing: Qwen / Qwen3-235B-A22B-FP8 ...
  -> Saved: ../output_clear/Qwen3-235B-A22B-FP8.csv
Analyzing: Qwen / Qwen2.5-72B-Instruct ...
  -> Saved: ../output_clear/Qwen2.5-72B-Instruct.csv
Analyzing: Qwen / Qwen2.5-14B-Instruct ...
  -> Saved: ../output_clear/Qwen2

In [8]:
import json
import glob
import os
import re
import csv




def load_disorder_map(json_path):
    if not os.path.exists(json_path):
        raise FileNotFoundError(f"disorder.json 파일을 찾을 수 없습니다: {json_path}")
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return {k: v["name"] for k, v in data.items() if "name" in v}


def parse_options(option_str):
    options = {}
    if not option_str:
        return options
    for line in option_str.strip().split('\n'):
        line = line.strip()
        if len(line) > 2 and line[1] == '.':
            options[line[0]] = line[2:].strip()
    return options


def count_target_in_file(filename, target_keyword):
    count = 0
    try:
        with open(filename, 'r', encoding='utf-8') as f:
            data = json.load(f)
        for _, entry in data.items():
            options = parse_options(entry.get('option', ''))
            response = entry.get('response', '') or ''
            for letter, disorder_name in options.items():
                if target_keyword.lower() in disorder_name.lower():
                    if letter in response:
                        count += 1
                    break
    except Exception as e:
        print(f"[Warning] {filename} 읽기 실패: {e}")
    return count


def process_file_group(work_dir, pattern, target_name):
    total = 0
    for fp in glob.glob(os.path.join(work_dir, pattern)):
        total += count_target_in_file(fp, target_name)
    return total


def analyze_type_dir(type_dir, disorder_map):
    parts = os.path.normpath(type_dir).split(os.sep)
    upper_code, lower_code = parts[-3], parts[-2]
    if not (re.match(r"D\d{3}", upper_code) and re.match(r"D\d{3}", lower_code)):
        return None

    upper_name = disorder_map.get(upper_code, "Unknown")
    lower_name = disorder_map.get(lower_code, "Unknown")

    return {
        "upper_code": upper_code,
        "upper_name": upper_name,
        "lower_code": lower_code,
        "lower_name": lower_name,
        "a_total": process_file_group(type_dir, "a_*.json", lower_name),
        "b_total": process_file_group(type_dir, "b_*.json", upper_name),
        "type_dir": type_dir,
    }


def analyze_all(high_dir, disorder_map, type_name="type4"):
    pattern = os.path.join(high_dir, "D???", "D???", type_name)
    rows = []
    for td in glob.glob(pattern):
        if os.path.isdir(td):
            res = analyze_type_dir(td, disorder_map)
            if res:
                rows.append(res)
    return rows, sum(r["a_total"] for r in rows), sum(r["b_total"] for r in rows)


def find_model_dirs(output_root):
    return [
        d for d in glob.glob(os.path.join(output_root, "*", "*"))
        if os.path.isdir(os.path.join(d, "high"))
    ]


def safe_model_name(model_path):
    return model_path.replace(os.sep, "__")


def save_pairs_csv(path, rows):
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["upper_code", "upper_name", "lower_code", "lower_name", "a_total", "b_total", "type_dir"]
        )
        writer.writeheader()
        writer.writerows(rows)


def save_summary_csv(path, model_name, pairs, a_total, b_total):
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["model", "pairs", "A_total", "B_total"]
        )
        writer.writeheader()
        writer.writerow({
            "model": model_name,
            "pairs": pairs,
            "A_total": a_total,
            "B_total": b_total
        })


def run_all_models_and_save(output_root, disorder_map, type_name="type4"):
    os.makedirs(OUTPUT_ANALYSIS_DIR, exist_ok=True)

    for model_dir in find_model_dirs(output_root):
        model_name = os.path.relpath(model_dir, output_root)
        safe_name = safe_model_name(model_name)
        high_dir = os.path.join(model_dir, "high")

        rows, sum_a, sum_b = analyze_all(high_dir, disorder_map, type_name)
        rows = sorted(rows, key=lambda r: (r["upper_code"], r["lower_code"]))

        pairs_csv = os.path.join(OUTPUT_ANALYSIS_DIR, f"{safe_name}__{type_name}_pairs.csv")
        # summary_csv = os.path.join(OUTPUT_ANALYSIS_DIR, f"{safe_name}__{type_name}_summary.csv")

        save_pairs_csv(pairs_csv, rows)
        # save_summary_csv(summary_csv, model_name, len(rows), sum_a, sum_b)

        # print(f"[Saved] {model_name} → {pairs_csv}, {summary_csv}")


if __name__ == "__main__":
    DISORDER_JSON_PATH = '../resources/knowledge_graph/EN/disorder.json'

    OUTPUT_ANALYSIS_DIR = "./type4_result_analysis"
    disorder_map = load_disorder_map(DISORDER_JSON_PATH)

    OUTPUT_ROOT = "../output_clear"
    run_all_models_and_save(OUTPUT_ROOT, disorder_map, type_name="type4")


# 3) Error analysis 1 (프롬프트에 따른 비교)

In [9]:
import pandas as pd

clear = pd.read_csv('mental_qa_type34_clear_results.csv').groupby(['Model', 'Type'])['Accuracy'].mean().unstack().sort_index()
single = pd.read_csv('mental_qa_type34_single_results.csv').groupby(['Model', 'Type'])['Accuracy'].mean().unstack().sort_index()
normal = pd.read_csv('mental_qa_type34_normal_results.csv').groupby(['Model', 'Type'])['Accuracy'].mean().unstack().sort_index()

print('Hybrid')
print(clear)
print("=======================================================")
print('Single')
print(single)
print("=======================================================")
print('Multiple')
print(normal)

FileNotFoundError: [Errno 2] No such file or directory: 'mental_qa_type34_clear_results.csv'

In [ ]:
raw_data = """Hybrid
Type                                   Type3      Type4
Model                                                  
Qwen2.5-72B-Instruct          52.750920  18.252414
Qwen3-235B-A22B           54.191839  28.628736
gemma-3-27b-it              52.505747   9.195632
Llama-3.1-70B-Instruct  39.754943  34.973218
MentaLLaMA-chat-13B         18.099540  16.168506
gpt-5.1                            22.482759  70.298736
gemini-2.5-pro              35.156897  59.816207
claude-sonnet-4.5        14.987931  74.843103

=======================================================
Single
Type                               Type3      Type4
Model                                              
Qwen2.5-14B-Instruct            0.0  81.708736
Qwen2.5-32B-Instruct            0.0  80.582529
Qwen2.5-72B-Instruct            0.0  80.758621
Qwen2.5-7B-Instruct             0.0  75.578276
Qwen3-14B                       0.0  71.440575
Qwen3-235B-A22B             0.0  80.835747
Qwen3-32B                       0.0  79.587126
Qwen3-8B                        0.0  74.950115
claude-haiku-4.5           NaN  85.708506
claude-sonnet-4.5          NaN  86.789540
gemini-2.5-flash              NaN  81.777816
gemini-2.5-pro                NaN  84.191724
gemma-3-12b-it                0.0  76.666437
gemma-3-27b-it                0.0  82.467241
gemma-3-4b-it                 0.0  67.203793
gpt-4o                               0.0  75.723793
gpt-5-mini                           0.0  83.034253
gpt-5.1                              0.0  87.578276
MentaLLaMA-chat-13B           0.0  48.391034
MentaLLaMA-chat-7B            0.0  31.103333
Llama-3.1-70B-Instruct    0.0  80.022874
Llama-3.1-8B-Instruct     0.0  71.041954
=======================================================
Multiple
Type                                   Type3      Type4
Model                                                  
Qwen2.5-14B-Instruct          47.631609  18.567701
Qwen2.5-32B-Instruct          44.444023  39.586092
Qwen2.5-72B-Instruct          47.341034  25.402414
Qwen2.5-7B-Instruct           44.505517  21.417931
Qwen3-14B                     51.754828   7.992414
Qwen3-235B-A22B           46.942759  40.268391
Qwen3-32B                     49.685402  22.896667
Qwen3-8B                      49.149195   6.229655
claude-haiku-4.5               NaN  45.448506
claude-sonnet-4.5              NaN  63.248621
gemini-2.5-flash                  NaN  35.111034
gemini-2.5-pro                    NaN  35.915632
gemma-3-12b-it              46.374943  19.226092
gemma-3-27b-it              45.762299  26.681724
gemma-3-4b-it               24.781264  30.352874
gpt-4o                             47.172299  37.969770
gpt-5-mini                         29.410000  66.727471
gpt-5.1                            50.176322  40.604828
MentaLLaMA-chat-13B         13.379310  24.789310
MentaLLaMA-chat-7B           2.390230  28.743103
Llama-3.1-70B-Instruct  51.157471  20.521149
Llama-3.1-8B-Instruct   35.172414  22.398161
"""

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. 원본 데이터 정의 (유저의 raw_data 변수가 있다고 가정)
raw_data = """Hybrid
Type                                   Type3      Type4
Model                                                  
Qwen2.5-72B-Instruct          52.750920  18.252414
Qwen3-235B-A22B           54.191839  28.628736
gemma-3-27b-it              52.505747   9.195632
Llama-3.1-70B-Instruct  39.754943  34.973218
MentaLLaMA-chat-13B         18.099540  16.168506
gpt-5.1                            22.482759  70.298736
gemini-2.5-pro              35.156897  59.816207
claude-sonnet-4.5        14.987931  74.843103

=======================================================
Single
Type                               Type3      Type4
Model                                              
Qwen2.5-14B-Instruct            0.0  81.708736
Qwen2.5-32B-Instruct            0.0  80.582529
Qwen2.5-72B-Instruct            0.0  80.758621
Qwen2.5-7B-Instruct             0.0  75.578276
Qwen3-14B                       0.0  71.440575
Qwen3-235B-A22B             0.0  80.835747
Qwen3-32B                       0.0  79.587126
Qwen3-8B                        0.0  74.950115
claude-haiku-4.5           NaN  85.708506
claude-sonnet-4.5          NaN  86.789540
gemini-2.5-flash              NaN  81.777816
gemini-2.5-pro                NaN  84.191724
gemma-3-12b-it                0.0  76.666437
gemma-3-27b-it                0.0  82.467241
gemma-3-4b-it                 0.0  67.203793
gpt-4o                               0.0  75.723793
gpt-5-mini                           0.0  83.034253
gpt-5.1                              0.0  87.578276
MentaLLaMA-chat-13B           0.0  48.391034
MentaLLaMA-chat-7B            0.0  31.103333
Llama-3.1-70B-Instruct    0.0  80.022874
Llama-3.1-8B-Instruct     0.0  71.041954
=======================================================
Multiple
Type                                   Type3      Type4
Model                                                  
Qwen2.5-14B-Instruct          47.631609  18.567701
Qwen2.5-32B-Instruct          44.444023  39.586092
Qwen2.5-72B-Instruct          47.341034  25.402414
Qwen2.5-7B-Instruct           44.505517  21.417931
Qwen3-14B                     51.754828   7.992414
Qwen3-235B-A22B           46.942759  40.268391
Qwen3-32B                     49.685402  22.896667
Qwen3-8B                      49.149195   6.229655
claude-haiku-4.5               NaN  45.448506
claude-sonnet-4.5              NaN  63.248621
gemini-2.5-flash                  NaN  35.111034
gemini-2.5-pro                    NaN  35.915632
gemma-3-12b-it              46.374943  19.226092
gemma-3-27b-it              45.762299  26.681724
gemma-3-4b-it               24.781264  30.352874
gpt-4o                             47.172299  37.969770
gpt-5-mini                         29.410000  66.727471
gpt-5.1                            50.176322  40.604828
MentaLLaMA-chat-13B         13.379310  24.789310
MentaLLaMA-chat-7B           2.390230  28.743103
Llama-3.1-70B-Instruct  51.157471  20.521149
Llama-3.1-8B-Instruct   35.172414  22.398161
"""
# 2. 파싱 함수 정의
def parse_data(text):
    blocks = text.split("=" * 55)
    records = []
    for block in blocks:
        lines = [l.strip() for l in block.strip().split('\n') if l.strip()]
        if not lines: continue
        
        category = lines[0]
        is_data = False
        for line in lines:
            if line.startswith("Model"):
                is_data = True
                continue
            if is_data:
                parts = line.split()
                if len(parts) >= 3:
                    records.append({
                        'Model': parts[0],
                        'Category': category,
                        'Type4': float(parts[-1]) if parts[-1] != 'NaN' else None
                    })
    return pd.DataFrame(records)

df = parse_data(raw_data)

# --- 순서 고정을 위한 로직 추가 ---
# 1. 'Hybrid' 카테고리에 있는 모델들을 데이터에 나타난 순서대로 추출합니다.
hybrid_order = df[df['Category'] == 'Hybrid']['Model'].unique().tolist()

# 3. 데이터 재구조화 (피벗)
pivot_df = df.pivot(index='Model', columns='Category', values='Type4')

# 4. 필터링: 'Hybrid'에 데이터가 있는 모델만 남기기
# 위에서 만든 hybrid_order를 기준으로 재정렬(reindex)하면 필터링과 순서 정렬이 동시에 됩니다.
pivot_df = pivot_df.reindex(hybrid_order)

# 5. X축 순서 설정
order = ['Multiple', 'Hybrid', 'Single']
pivot_df = pivot_df.reindex(columns=order)

# 6. 시각화
plt.figure(figsize=(6, 5))

for model_name in pivot_df.index:
    # gpt / gemini / claude 계열은 점선
    if any(k in model_name.lower() for k in ['gpt', 'gemini', 'claude']):
        linestyle = '--'
    else:
        linestyle = '-'

    plt.plot(
        order,
        pivot_df.loc[model_name],
        marker='o',
        label=model_name,
        linewidth=2,
        linestyle=linestyle
    )
# plt.title('Type4 Score Trend', fontsize=15)
plt.xlabel('Evaluation Type', fontsize=12)
plt.ylabel('Type4 Acc', fontsize=12)

# 범례 순서는 plot을 호출한 순서대로 쌓이므로, pivot_df.index 순서와 동일해집니다.
plt.legend(bbox_to_anchor=(0, 1), loc='upper left', fontsize='x-small')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# 4) Error type analysis (답변 오답 유형 비교)

In [ ]:
import json
import glob
import os
import re
import csv
from collections import defaultdict

# -----------------------------------------------------------------------------
# 설정: 경로 지정
# -----------------------------------------------------------------------------


def load_disorder_map(json_path):
    if not os.path.exists(json_path):
        raise FileNotFoundError(f"disorder.json 파일을 찾을 수 없습니다: {json_path}")
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return {k: v["name"] for k, v in data.items() if "name" in v}


def extract_answer(text):
    if not isinstance(text, str): return set()
    text = text.upper()
    
    # 1. Remove common prefixes
    text = re.sub(r'^(?:ANSWER|OUTPUT|SELECTION|THE ANSWER IS)[:\s-]*', '', text.strip())

    # 2. [Core Regex] Find the first valid answer sequence.
    match = re.search(r'([A-D](?:\s*(?:,|&|and)\s*[A-D])*)', text)

    if match:
        return set(re.findall(r'[A-D]', match.group(1)))
    
    return set()


def generate_gt(text):
    """
    Extracts the option letters from the Ground Truth (GT) text.
    """
    if not isinstance(text, str):
        return set()
    
    text = text.upper().strip()
    
    # 1. Split the text at the first occurrence of either a period (.) or a closing parenthesis ())
    text = re.split(r'[.)]', text, 1)[0]
        
    # 2. Extract option letters (A, B, C, D) from the prefix.
    return set(re.findall(r'\b[A-D]\b', text))


def analyze_files_stats(file_pattern):
    """
    특정 파일 패턴(예: type3/a_*.json)에 대해 통계 집계
    Changes: Uses extract_answer and generate_gt for Exact Match logic.
    """
    stats = {
        "correct": 0,
        "incorrect": 0,
        "incorrect_2choices": 0,      # 오답이면서 2개 고름
        "correct_1choice": 0,         # 정답이면서 1개 고름
        "correct_3plus_choices": 0,   # 정답이면서 3개 이상 고름
        "correct_2plus_choices": 0    # 정답이면서 2개 이상 고름
    }

    files = glob.glob(file_pattern)
    for filename in files:
        try:
            with open(filename, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            for _, entry in data.items():
                response = entry.get('response', '') or ''
                answer = entry.get('answer', '') or ''
                
                res_answer_set = extract_answer(response)
                gt_set = generate_gt(answer)
                
                num_selected = len(res_answer_set)

                # 3. 정답/오답 판단
                # New: Exact Match (gt_set == res_answer_set)
                is_correct = (gt_set == res_answer_set and len(gt_set) > 0)

                if is_correct:
                    stats["correct"] += 1
                else:
                    stats["incorrect"] += 1
                    if num_selected == 1:
                        stats["correct_1choice"] += 1
                    if num_selected >= 2:
                        stats["correct_2plus_choices"] += 1
                    if num_selected >= 3:
                        stats["correct_3plus_choices"] += 1
                    if num_selected == 2:
                        stats["incorrect_2choices"] += 1

        except Exception as e:
            print(f"[Warning] {filename} 분석 중 에러: {e}")
            
    return stats


def process_pair_directory(pair_dir, disorder_map):
    """
    D001/D005 와 같은 질환 쌍 디렉토리 하나를 처리 (Type3, Type4 모두)
    """
    parts = os.path.normpath(pair_dir).split(os.sep)
    # 경로 끝부분이 Dxxx/Dyyy 라고 가정
    try:
        upper_code, lower_code = parts[-2], parts[-1] 
    except IndexError:
        return None
    
    if not (re.match(r"D\d{3}", upper_code) and re.match(r"D\d{3}", lower_code)):
        return None

    upper_name = disorder_map.get(upper_code, "Unknown")
    lower_name = disorder_map.get(lower_code, "Unknown")

    # 결과 저장용 딕셔너리 초기화
    row = {
        "upper_code": upper_code,
        "upper_name": upper_name,
        "lower_code": lower_code,
        "lower_name": lower_name,
        
        # Type 3 Stats
        "type3_correct": 0,
        "type3_incorrect": 0,
        "type3_incorrect_2choices": 0,
        "type3_incorrect_1choice": 0,
        "type3_incorrect_3plus_choices": 0,

        # Type 4 Stats
        "type4_correct": 0,
        "type4_incorrect": 0,
        "type4_incorrect_1_choices": 0,
        "type4_incorrect_2plus_choices": 0
    }

    # --- Type 3 분석 ---
    t3_dir = os.path.join(pair_dir, "type3")
    if os.path.isdir(t3_dir):
        # a_*.json (Target: Lower Name)
        s_a = analyze_files_stats(os.path.join(t3_dir, "*.json"))

        # 합산
        row["type3_correct"] = s_a["correct"]
        row["type3_incorrect"] = s_a["incorrect"]
        row["type3_incorrect_2choices"] = s_a["incorrect_2choices"]
        row["type3_incorrect_1choice"] = s_a["correct_1choice"]
        row["type3_incorrect_3plus_choices"] = s_a["correct_3plus_choices"]

    # --- Type 4 분석 ---
    t4_dir = os.path.join(pair_dir, "type4")
    if os.path.isdir(t4_dir):
        # a_*.json (Target: Lower Name)
        s_a = analyze_files_stats(os.path.join(t4_dir, "a_*.json"))
        # b_*.json (Target: Upper Name)
        s_b = analyze_files_stats(os.path.join(t4_dir, "b_*.json"))

        # 합산
        row["type4_correct"] = s_a["correct"] + s_b["correct"]
        row["type4_incorrect"] = s_a["incorrect"] + s_b["incorrect"]

        row["type4_incorrect_1_choices"] = s_a["correct_1choice"] + s_b["correct_1choice"]
        row["type4_incorrect_2plus_choices"] = s_a["correct_2plus_choices"] + s_b["correct_2plus_choices"]

    return row


def analyze_model_high_dir(high_dir, disorder_map):
    """
    high 폴더 아래의 모든 Dxxx/Dyyy 쌍을 찾아서 분석
    """
    # high/D???/D??? 패턴 검색
    pattern = os.path.join(high_dir, "D???", "D???")
    rows = []
    
    for pair_dir in glob.glob(pattern):
        if os.path.isdir(pair_dir):
            res = process_pair_directory(pair_dir, disorder_map)
            if res:
                rows.append(res)
    return rows


def find_model_dirs(output_root):
    # output_root/{ModelName}/high 가 존재하는지 확인
    return [
        d for d in glob.glob(os.path.join(output_root, "*", "*"))
        if os.path.isdir(os.path.join(d, "high"))
    ]


def save_csv(path, rows):
    if not rows:
        return
    
    # 컬럼 순서 지정
    fieldnames = [
        "upper_code", "upper_name", "lower_code", "lower_name",
        "type3_correct",
        "type3_incorrect",
        "type4_correct",
        "type4_incorrect",
        "type3_incorrect_2choices",      # type3 오답이면서 2개 고름
        "type3_incorrect_1choice",         # type3 정답이면서 1개 고름
        "type3_incorrect_3plus_choices",   # type3 정답이면서 3~4개 고름
        "type4_incorrect_1_choices",   # type4 정답이면서 2개 이상 고름
        "type4_incorrect_2plus_choices",   # type4 정답이면서 2개 이상 고름
    ]
    
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def run_all_models_and_save(output_root, disorder_map):
    # 저장 경로 생성
    os.makedirs(OUTPUT_ANALYSIS_DIR, exist_ok=True)
    print(f"Results will be saved to: {OUTPUT_ANALYSIS_DIR}")

    model_dirs = find_model_dirs(output_root)
    print(f"총 {len(model_dirs)}개의 모델 디렉토리를 찾았습니다.")

    for model_dir in model_dirs:
        # [수정됨] relpath 대신 basename을 사용하여 최하위 폴더 이름만 가져옴
        # 예: .../gpt/gpt-5.1 -> gpt-5.1
        model_name = os.path.basename(model_dir) 
        
        high_dir = os.path.join(model_dir, "high")

        print(f"Processing: {model_name} ...")
        
        rows = analyze_model_high_dir(high_dir, disorder_map)
        
        # 정렬: Upper Code -> Lower Code 순
        rows = sorted(rows, key=lambda r: (r["upper_code"], r["lower_code"]))

        # [수정됨] 상위 디렉토리명이 포함되지 않은 model_name을 바로 사용
        csv_filename = f"{model_name}__analysis.csv"
        csv_path = os.path.join(OUTPUT_ANALYSIS_DIR, csv_filename)

        save_csv(csv_path, rows)
        print(f"  -> Saved to {csv_path}")



if __name__ == "__main__":
    DISORDER_JSON_PATH = '../resources/knowledge_graph/EN/disorder.json'
    OUTPUT_ROOT = "../output_clear"
    OUTPUT_ANALYSIS_DIR = '../scripts/type3_4_result_analysis'

    disorder_map = load_disorder_map(DISORDER_JSON_PATH)
    run_all_models_and_save(OUTPUT_ROOT, disorder_map)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import glob
import os

# 분석할 컬럼들
cols = [
    "type3_incorrect_2choices",
    "type3_incorrect_1choice",
    "type3_incorrect_3plus_choices",
    "type4_incorrect_1_choices",
    "type4_incorrect_2plus_choices",
]

results = []

# CSV 파일 경로 패턴
csv_pattern = "/home/user/hgyoo/mental/MentalQA_v1/scripts/type3_4_result_analysis/*__analysis.csv"
csv_files = glob.glob(csv_pattern)

print(f"발견된 파일 개수: {len(csv_files)}")

for csv_path in csv_files:
    # 1. CSV 읽기
    df = pd.read_csv(csv_path)

    # 2. NaN 처리
    df[cols] = df[cols].fillna(0)

    # 3. 모델 전체 합계 계산 (Raw Count)
    model_stats = df[cols].sum().to_frame().T

    # 4. 모델 이름 추출 및 추가
    model_name = os.path.basename(csv_path).replace("__analysis.csv", "")
    model_stats.insert(0, 'model', model_name)

    results.append(model_stats)

# 결과 합치기
final_df = pd.concat(results, ignore_index=True)

# ---------------------------------------------------------
# [수정된 부분] 요청하신 컬럼 병합 및 비율 계산 로직
# ---------------------------------------------------------

# 1. 컬럼 병합 (Raw Count 상태에서 더하기)
# final_df['correct'] = final_df['type3_correct'] + final_df['type4_correct']
final_df['incorrect'] = final_df['type3_incorrect_2choices'] + final_df['type4_incorrect_1_choices']
final_df['under'] = final_df['type3_incorrect_1choice']
final_df['over'] = final_df['type3_incorrect_3plus_choices'] + final_df['type4_incorrect_2plus_choices']

# 2. 필요한 컬럼만 선택
display_cols = ['model', 'incorrect', 'under', 'over']
final_df = final_df[display_cols]

# 3. 전체 합계 19575로 나누어 비율(%) 계산
# (Type3: 6525 + Type4: 13050 = 19575)
total_count = 19575
target_cols = ['incorrect', 'under', 'over']

for col in target_cols:
    final_df[col] = (final_df[col] * 100 / total_count).round(2)

# 모델명 기준 정렬
final_df = final_df.sort_values(by='model', ascending=False)


target_models = [
    'Qwen2.5-72B-Instruct',
    'Qwen3-235B-A22B-FP8',
    'gemma-3-27b-it',
    "Llama-3.1-70B-Instruct",
    'MentaLLaMA-chat-13B',
    "gpt-5.1",
    "gemini-2.5-pro",
    'claude-sonnet-4.5'
]

# 1. 필터링
subset_df = final_df[final_df["model"].isin(target_models)]

# 2. target_models 순서대로 정렬
plot_df = (
    subset_df
    .set_index("model")
    .reindex(target_models)
)

fig, ax = plt.subplots(figsize=(8, 5))

plot_df.plot(kind="bar", ax=ax)

ax.set_ylabel("Percentage (%)")
ax.set_title("Error analysis")
ax.set_xticklabels(ax.get_xticklabels(), rotation=45)
ax.legend(bbox_to_anchor=(0.82, 1), loc="upper left")
ax.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import glob
import os
import numpy as np
from matplotlib.patches import Patch

# ---------------------------------------
# 1) 설정
# ---------------------------------------
type3_cols = [
    "type3_incorrect_2choices",
    "type3_incorrect_1choice",
    "type3_incorrect_3plus_choices",
]
type4_cols = [
    "type4_incorrect_1_choices",
    "type4_incorrect_2plus_choices",
]
cols = type3_cols + type4_cols

csv_pattern = "/home/user/hgyoo/mental/MentalQA_v1/scripts/type3_4_result_analysis/*__analysis.csv"
csv_files = glob.glob(csv_pattern)

print(f"발견된 파일 개수: {len(csv_files)}")

target_models = [
    "Qwen2.5-72B-Instruct",
    "Qwen3-235B-A22B-FP8",
    "gemma-3-27b-it",
    "gpt-5.1",
    "claude-sonnet-4.5",
    "gemini-2.5-pro",
]

# Type3: 6525 + Type4: 13050 = 19575 (기존 기준 유지)
total_count = 19575

# ---------------------------------------
# 2) CSV들 읽어서 모델별 합계 만들기
# ---------------------------------------
results = []

for csv_path in csv_files:
    df = pd.read_csv(csv_path)

    # 누락 컬럼 보정 + NaN -> 0
    for c in cols:
        if c not in df.columns:
            df[c] = 0
    df[cols] = df[cols].fillna(0)

    # 모델별 raw 합계
    model_stats = df[cols].sum().to_frame().T

    model_name = os.path.basename(csv_path).replace("__analysis.csv", "")
    model_stats.insert(0, "model", model_name)

    results.append(model_stats)

if not results:
    raise RuntimeError("분석할 CSV가 없습니다. 경로/패턴을 확인하세요.")

final_df = pd.concat(results, ignore_index=True)

# ---------------------------------------
# 3) 타겟 모델만 뽑고, 순서 정렬
# ---------------------------------------
subset_df = final_df[final_df["model"].isin(target_models)].copy()
plot_df = subset_df.set_index("model").reindex(target_models).fillna(0)

plot_df = plot_df.rename(
    index={
        "Qwen3-235B-A22B-FP8": "Qwen3-235B",
        "Qwen2.5-72B-Instruct": "Qwen2.5-72B"
    }
)
# ---------------------------------------
# 4) % 변환 (raw로 두고 싶으면 이 블록 제거)
# ---------------------------------------
TYPE3_TOTAL = 6525
TYPE4_TOTAL = 13050

for c in type3_cols:
    plot_df[c] = (plot_df[c] * 100 / TYPE3_TOTAL).round(2)

for c in type4_cols:
    plot_df[c] = (plot_df[c] * 100 / TYPE4_TOTAL).round(2)

# ---------------------------------------
# 5) Plot
#   - Type3 bar(왼쪽) / Type4 bar(오른쪽) 사이에 gap 추가
#   - bar 아래에 Type3 / Type4 텍스트 표시
#   - legend는 Incorrect / Under / Over만
# ---------------------------------------
# 의미 기반 색상 매핑
COLOR_INCORRECT = "#d62728"  # red
COLOR_UNDER     = "#ff7f0e"  # orange
COLOR_OVER      = "#1f77b4"  # blue

x = np.arange(len(plot_df.index))

bar_w = 0.2   # bar 폭
gap   = 0.2   # Type3와 Type4 사이 간격(각 모델 내)
offset = (bar_w / 2) + (gap / 2)

x_type3 = x - offset
x_type4 = x + offset

fig, ax = plt.subplots(figsize=(6, 4))

# --- Type3 stacked (왼쪽 bar) ---
bottom = np.zeros(len(plot_df))
ax.bar(x_type3, plot_df["type3_incorrect_2choices"], bar_w, bottom=bottom, color=COLOR_INCORRECT)
bottom += plot_df["type3_incorrect_2choices"]

ax.bar(x_type3, plot_df["type3_incorrect_1choice"], bar_w, bottom=bottom, color=COLOR_UNDER)
bottom += plot_df["type3_incorrect_1choice"]

ax.bar(x_type3, plot_df["type3_incorrect_3plus_choices"], bar_w, bottom=bottom, color=COLOR_OVER)

# --- Type4 stacked (오른쪽 bar) ---
bottom = np.zeros(len(plot_df))
ax.bar(x_type4, plot_df["type4_incorrect_1_choices"], bar_w, bottom=bottom, color=COLOR_INCORRECT)
bottom += plot_df["type4_incorrect_1_choices"]

ax.bar(x_type4, plot_df["type4_incorrect_2plus_choices"], bar_w, bottom=bottom, color=COLOR_OVER)

# ---------------------------------------
# 6) X축: 모델명은 가운데, bar 아래에 Type3/Type4 텍스트
# ---------------------------------------
ax.set_xticks(x)
ax.set_xticklabels(plot_df.index, rotation=30, ha="right")

# bar 아래에 Type3/Type4 표기(축 좌표계 사용)
for i in range(len(x)):
    ax.text(x_type3[i], -0.01, "Type3", transform=ax.get_xaxis_transform(),
            ha="center", va="top", fontsize=8)
    ax.text(x_type4[i], -0.01, "Type4", transform=ax.get_xaxis_transform(),
            ha="center", va="top", fontsize=8)

# 아래쪽 여백 확보(텍스트가 잘리지 않게)
plt.subplots_adjust(bottom=0.28)

# ---------------------------------------
# 7) Styling + Legend(의미별 3개만)
# ---------------------------------------
ax.set_ylabel("Percentage (%)")
# ax.set_title("Type3 / Type4 Error")
ax.grid(axis="y", linestyle="--", alpha=0.6)

legend_handles = [
    Patch(facecolor=COLOR_INCORRECT, label="Incorrect"),
    Patch(facecolor=COLOR_UNDER, label="Under"),
    Patch(facecolor=COLOR_OVER, label="Over"),
]
ax.legend(handles=legend_handles, bbox_to_anchor=(0.75, 1), loc="upper left")

plt.tight_layout()
plt.show()


In [ ]:
plot_df

# 5) (가장 어려운 분석) 익숙한 질병에 대한 편중

In [ ]:
import json
import glob
import os
import re

# -----------------------------------------------------------------------------
# 1. 설정 및 데이터 로드 함수
# -----------------------------------------------------------------------------

def load_disorder_map(json_path):
    if not os.path.exists(json_path):
        raise FileNotFoundError(f"disorder.json 파일을 찾을 수 없습니다: {json_path}")
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return {k: v["name"].strip().lower() for k, v in data.items() if "name" in v}

def parse_option_string(option_str):
    if not isinstance(option_str, str):
        return {}
    options = {}
    pattern = re.compile(r'\s*([A-D])[.)]\s*(.+)')
    lines = option_str.split('\n')
    for line in lines:
        match = pattern.search(line)
        if match:
            key, name = match.groups()
            options[key] = name.strip().lower()
    return options

def extract_answer_keys(text):
    if not isinstance(text, str): return set()
    text = text.upper()
    text = re.sub(r'^(?:ANSWER|OUTPUT|SELECTION|THE ANSWER IS)[:\s-]*', '', text.strip())
    match = re.search(r'([A-D](?:\s*(?:,|&|and)\s*[A-D])*)', text)
    if match:
        return set(re.findall(r'[A-D]', match.group(1)))
    return set()

# -----------------------------------------------------------------------------
# 2. Type 4 타겟(Strict Target) 불일치 분석 함수
# -----------------------------------------------------------------------------

def analyze_type4_strict_target(base_dir, disorder_map):
    
    name_to_code_map = {name: code for code, name in disorder_map.items()}

    stats = {
        "files_processed": 0,
        "total_questions": 0,
        "incorrect_sample_count": 0, 
        "samples": [] 
    }

    search_pattern = os.path.join(base_dir, "D*", "D*", "type4", "[ab]_*.json")
    files = glob.glob(search_pattern)
    
    print(f"검색 경로: {search_pattern}")
    print(f"분석 대상 파일 수: {len(files)} 개")

    for filepath in files:
        stats["files_processed"] += 1
        
        parts = os.path.normpath(filepath).split(os.sep)
        try:
            upper_code = parts[-4]
            lower_code = parts[-3]
            filename = parts[-1]
        except IndexError:
            continue

        if not (upper_code.startswith("D") and lower_code.startswith("D")):
            continue

        upper_name = disorder_map.get(upper_code, "").lower()
        lower_name = disorder_map.get(lower_code, "").lower()

        # [수정됨 1] Target Code 변수 추가 설정
        if filename.startswith("a_"):
            valid_target_name = upper_name
            valid_target_code = upper_code  # <--- 추가
            file_type_label = "a (Target: Upper)"
        elif filename.startswith("b_"):
            valid_target_name = lower_name
            valid_target_code = lower_code  # <--- 추가
            file_type_label = "b (Target: Lower)"
        else:
            continue

        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                data = json.load(f)
        except Exception as e:
            print(f"[Error] 파일 로드 실패 ({filepath}): {e}")
            continue

        for q_id, entry in data.items():
            stats["total_questions"] += 1
            
            option_str = entry.get('option', '')
            response_str = entry.get('response', '')
            
            option_map = parse_option_string(option_str)
            selected_keys = extract_answer_keys(response_str)
            
            incorrect_selections = []
            
            for char in selected_keys:
                selected_name = option_map.get(char, "").strip().lower()
                
                if not selected_name:
                    continue
                
                # 비교 로직은 여전히 Name을 사용 (정확한 매칭을 위해)
                is_correct_target = (selected_name == valid_target_name)
                
                if not is_correct_target:
                    selected_code = name_to_code_map.get(selected_name, "Unknown")
                    incorrect_selections.append({
                        "char": char,
                        "name": selected_name,
                        "code": selected_code
                    })

            if len(incorrect_selections) > 0:
                stats["incorrect_sample_count"] += 1
                
                # [수정됨 2] target_name 대신 target_code 저장
                stats["samples"].append({
                    "file": filename,
                    "pair": f"{upper_code} vs {lower_code}",
                    "file_type": file_type_label,
                    "q_id": q_id,
                    "response": response_str,
                    "target_code": valid_target_code, # <--- 변경됨 (Name -> Code)
                    "incorrect_details": incorrect_selections
                })

    return stats

# -----------------------------------------------------------------------------
# 3. 메인 실행부
# -----------------------------------------------------------------------------

if __name__ == "__main__":
    DISORDER_JSON_PATH = "../resources/knowledge_graph/EN/disorder.json"
    TARGET_FOLDER = "/home/user/hgyoo/mental/MentalQA_v1/output_clear"
    OUTPUT_DIR = "./type4_analysis_results"

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # disorder map 로드
    d_map = load_disorder_map(DISORDER_JSON_PATH)

    # provider 단위 순회 (anthropic, gpt, google, ...)
    for provider in os.listdir(TARGET_FOLDER):
        provider_path = os.path.join(TARGET_FOLDER, provider)
        if not os.path.isdir(provider_path):
            continue

        # model 단위 순회 (gpt-5.1, gemini-2.5-pro, ...)
        for model_name in os.listdir(provider_path):
            model_path = os.path.join(provider_path, model_name)
            if not os.path.isdir(model_path):
                continue

            high_path = os.path.join(model_path, "high")
            if not os.path.isdir(high_path):
                continue

            print(f"[INFO] Processing: {provider}/{model_name}")

            # 분석 실행
            result = analyze_type4_strict_target(high_path, d_map)

            # 결과 저장
            output_path = os.path.join(
                OUTPUT_DIR,
                f"{model_name}_type4_analysis.jsonl"
            )
            with open(output_path, "w", encoding="utf-8") as f:
                json.dump(result, f, ensure_ascii=False, indent=2)

    print("[DONE] All models processed.")
        


In [ ]:
result

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
from collections import Counter
import numpy as np

# 1. 데이터 준비
data = result['samples']

# 2. 엣지 및 노드 추출
edges = []
left_nodes = set()
right_nodes = set()
labels = {}

for item in data:
    target_original = item['target_code']
    target_node = f"{target_original}_L"

    left_nodes.add(target_node)
    labels[target_node] = target_original

    for detail in item['incorrect_details']:
        incorrect_original = detail['code']
        incorrect_node = f"{incorrect_original}_R"

        right_nodes.add(incorrect_node)
        labels[incorrect_node] = incorrect_original

        edges.append((target_node, incorrect_node))

# 3. 그래프 생성 (edge weight = 빈도)
edge_counts = Counter(edges)

B = nx.Graph()
B.add_nodes_from(left_nodes, bipartite=0)
B.add_nodes_from(right_nodes, bipartite=1)

for (u, v), count in edge_counts.items():
    B.add_edge(u, v, weight=count)

# 4. edge weight 수집 & 상위 50% 기준
all_weights = [attr["weight"] for _, _, attr in B.edges(data=True)]
threshold = np.percentile(all_weights, 70)

# 👉 draw할 edge만 선택
selected_edges = [
    (u, v) for u, v, attr in B.edges(data=True)
    if attr["weight"] >= threshold
]

# 5. 좌/우 노드 정렬 및 좌표
left_sorted = sorted(left_nodes)
right_sorted = sorted(right_nodes)

pos = {}

for i, node in enumerate(left_sorted):
    y = 1 - (i / (len(left_sorted) - 1)) if len(left_sorted) > 1 else 0.5
    pos[node] = (-1, y)

for i, node in enumerate(right_sorted):
    y = 1 - (i / (len(right_sorted) - 1)) if len(right_sorted) > 1 else 0.5
    pos[node] = (1, y)

# 6. edge width 계산 (선택된 edge만)
selected_weights = [B[u][v]["weight"] for u, v in selected_edges]

min_width = 0.8
max_width = 5.0
max_weight = max(selected_weights) if selected_weights else 1

edge_widths = [
    min_width + (w / max_weight) * (max_width - min_width)
    for w in selected_weights
]

# 7. 시각화
fig_height = max(10, max(len(left_sorted), len(right_sorted)) * 0.4)
plt.figure(figsize=(12, fig_height))

# 노드 (전부 출력)
nx.draw_networkx_nodes(
    B, pos,
    nodelist=left_sorted,
    node_color='skyblue',
    node_size=800,
    alpha=0.9
)

nx.draw_networkx_nodes(
    B, pos,
    nodelist=right_sorted,
    node_color='lightcoral',
    node_size=800,
    alpha=0.9
)

# edge (상위 50%만)
nx.draw_networkx_edges(
    B, pos,
    edgelist=selected_edges,
    width=edge_widths,
    edge_color='gray',
    alpha=0.5
)

# 라벨
nx.draw_networkx_labels(B, pos, labels=labels, font_size=9)

plt.text(-1.2, 1.05, "Target Code (Answer)", fontsize=14, ha='center', fontweight='bold')
plt.text(1.2, 1.05, "Incorrect Code (Prediction)", fontsize=14, ha='center', fontweight='bold')

plt.xlim(-1.5, 1.5)
plt.ylim(-0.1, 1.1)
plt.axis('off')
plt.title("Confusion Pattern Analysis (Top 50% Edges Only)", fontsize=16)

plt.tight_layout()
plt.show()
